# 6. Local Email Reply Assistant

## What We're Building
An assistant that reads incoming emails, classifies their intent, and drafts
appropriate replies — all using structured output parsing with Pydantic.

**You will learn:**
- Intent classification with LLMs
- Structured output with Pydantic models
- Conditional logic based on classification
- Professional email drafting

**Stack:** Ollama, LangChain, Pydantic

In [ ]:
# !pip install -q langchain langchain-ollama pydantic

from langchain_ollama import ChatOllama
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from pydantic import BaseModel, Field
from typing import Literal

llm = ChatOllama(model="qwen3:8b", temperature=0.2)
print("LLM ready!")

## Step 1 — Define the Email Classification Schema

In [ ]:
# Define structured output using Pydantic
class EmailClassification(BaseModel):
    """Classification of an incoming email."""
    intent: Literal["question", "complaint", "request", "info", "urgent"] = Field(
        description="The primary intent of the email"
    )
    sentiment: Literal["positive", "neutral", "negative"] = Field(
        description="Overall sentiment of the email"
    )
    priority: Literal["high", "medium", "low"] = Field(
        description="Priority level based on urgency and content"
    )
    key_points: list[str] = Field(
        description="Main points or questions from the email (2-4 items)"
    )
    suggested_tone: Literal["formal", "friendly", "empathetic", "urgent"] = Field(
        description="Recommended tone for the reply"
    )

print("Schema defined!")
print(f"Fields: {list(EmailClassification.model_fields.keys())}")

## Step 2 — Sample Emails

In [ ]:
sample_emails = [
    {
        "from": "client@example.com",
        "subject": "Urgent: API Integration Issues",
        "body": """Hi Team,

We've been experiencing intermittent failures with the payment API since
yesterday. Our checkout flow is breaking for about 30% of customers.
This is impacting our revenue significantly. We need this resolved ASAP.

Can someone look into this immediately? We're losing approximately
$5,000/hour in failed transactions.

Thanks,
Mark"""
    },
    {
        "from": "prospect@company.com",
        "subject": "Question about Enterprise Plan",
        "body": """Hello,

I came across your product and I'm interested in the enterprise plan.
Could you let me know:
1. What's the pricing for 500+ users?
2. Do you offer SSO integration?
3. Is there a trial period?

Looking forward to hearing from you.

Best regards,
Lisa"""
    },
    {
        "from": "partner@business.com",
        "subject": "Partnership Proposal Follow-up",
        "body": """Hi,

Just following up on our conversation from last week about the potential
integration partnership. We've prepared a technical spec document and would
love to schedule a call to discuss next steps.

Would Thursday or Friday afternoon work for a 30-minute call?

Cheers,
David"""
    },
]

print(f"Loaded {len(sample_emails)} sample emails")

## Step 3 — Classify Emails

In [ ]:
import json

classify_prompt = ChatPromptTemplate.from_template(
    """Analyze this email and classify it. Return ONLY a valid JSON object.

FROM: {sender}
SUBJECT: {subject}
BODY:
{body}

Return JSON with these exact fields:
- "intent": one of "question", "complaint", "request", "info", "urgent"
- "sentiment": one of "positive", "neutral", "negative"
- "priority": one of "high", "medium", "low"
- "key_points": list of 2-4 main points as strings
- "suggested_tone": one of "formal", "friendly", "empathetic", "urgent"

JSON only, no markdown, no explanation:"""
)

json_parser = JsonOutputParser(pydantic_object=EmailClassification)
classify_chain = classify_prompt | llm | json_parser

# Classify each email
classifications = []
for email in sample_emails:
    print(f"\nClassifying: {email['subject']}")
    result = classify_chain.invoke({
        "sender": email["from"],
        "subject": email["subject"],
        "body": email["body"],
    })
    classifications.append(result)
    print(f"  Intent: {result.get('intent')} | Priority: {result.get('priority')}")
    print(f"  Sentiment: {result.get('sentiment')} | Tone: {result.get('suggested_tone')}")
    print(f"  Key points: {result.get('key_points')}")

## Step 4 — Draft Replies Based on Classification

In [ ]:
reply_prompt = ChatPromptTemplate.from_template(
    """Draft a reply to this email based on the classification below.

ORIGINAL EMAIL:
From: {sender}
Subject: {subject}
Body: {body}

CLASSIFICATION:
Intent: {intent}
Priority: {priority}
Suggested Tone: {tone}
Key Points: {key_points}

Guidelines:
- Use the suggested tone
- Address each key point
- If urgent/complaint: acknowledge the issue, provide timeline
- If question: answer clearly and offer to elaborate
- If request: acknowledge and suggest next steps
- Keep it professional and concise (under 200 words)
- Sign off as "Best regards, Support Team"

Draft the reply:"""
)

reply_chain = reply_prompt | llm | StrOutputParser()

# Generate replies for all classified emails
for email, classification in zip(sample_emails, classifications):
    print(f"\n{'='*60}")
    print(f"RE: {email['subject']}")
    print(f"{'='*60}")

    reply = reply_chain.invoke({
        "sender": email["from"],
        "subject": email["subject"],
        "body": email["body"],
        "intent": classification.get("intent", "info"),
        "priority": classification.get("priority", "medium"),
        "tone": classification.get("suggested_tone", "professional"),
        "key_points": str(classification.get("key_points", [])),
    })
    print(reply)

In [ ]:
# Full email processing pipeline
def process_email(sender: str, subject: str, body: str) -> dict:
    """Classify an email and draft a reply."""
    # Classify
    classification = classify_chain.invoke({
        "sender": sender, "subject": subject, "body": body,
    })

    # Draft reply
    reply = reply_chain.invoke({
        "sender": sender, "subject": subject, "body": body,
        "intent": classification.get("intent", "info"),
        "priority": classification.get("priority", "medium"),
        "tone": classification.get("suggested_tone", "professional"),
        "key_points": str(classification.get("key_points", [])),
    })

    return {"classification": classification, "reply": reply}

# Quick test
result = process_email(
    "user@example.com",
    "Feature Request: Dark Mode",
    "Hi, I love your product! Would it be possible to add a dark mode option?"
)
print("Classification:", json.dumps(result["classification"], indent=2))
print("\nDraft Reply:\n", result["reply"])

## Summary & Next Steps

**What you learned:**
- ✅ Pydantic models for structured LLM output
- ✅ JSON output parsing with LangChain
- ✅ Intent classification → conditional generation
- ✅ Multi-step processing pipelines

**Next steps:**
- Add email threading / conversation context
- Integrate with IMAP for real email ingestion
- Build a priority inbox dashboard

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3.5:9b", temperature=0.1)

prompt = PromptTemplate.from_template(
    "You are a helpful local AI assistant. Answer the user's question:\n\nQuestion: {question}\n\nAnswer:"
)

chain = prompt | llm | StrOutputParser()

# response = chain.invoke({"question": "What can you help me with?"})
# print(response)
print("LangChain inference pipeline ready!")
